### Stream Customer data from cloud files to delta lake 
- 1.Read files using DataStreamReader API
- 2.Transfer the dataframe to add the following columns a) Filepath b)ingestion date
- 3.Write the transform data stream to delta lake

1.Read files using DataStreamReader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

Customer_schema = StructType(
    fields = [StructField("customer_id", IntegerType()),
              StructField("customer_name", StringType()),
              StructField("date_of_birth", DateType()),
              StructField("telephone", StringType()),
              StructField("email", StringType()),
              StructField("member_since", DateType()),
              StructField("created_timestamp",TimestampType())
              ]
)

In [0]:
customer_df = (spark.readStream
                    .format("json")
                    .schema(Customer_schema)
                    .load("/Volumes/gizmobox/landing/operational_data/Customer_Stream/")
)
    

2.Transfer the dataframe to add the following columns a) Filepath b)ingestion date

In [0]:
from pyspark.sql.functions import current_timestamp, col

customer_tranformed_df = (customer_df.withColumn("file_path",col("_metadata.file_path"))
                                     .withColumn("ingestion_date",current_timestamp())
                                     )

3.Write the transform data stream to delta lake

In [0]:
streaming_query = (customer_tranformed_df.writeStream
  .format("delta")
  .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/Customer_Stream/_checkpoints_stream")
  .toTable("gizmobox.bronze.customer_stream")
)


In [0]:
streaming_query.stop()

In [0]:
%sql
select * from gizmobox.bronze.customer_stream